# validating SBI
We'll implement different SBI validation techniques such as simulation-based calibration, probability-probability plots, and TARP 

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import corner

In [ ]:
import torch
import sbi
from sbi.inference import NPE

In [ ]:
np.random.seed(1)

# Example 1
Let's go straight into the example of fitting a line to some data. This is the same example as previous notebooks.

In [ ]:
a_true, b_true = 0, 1.75
x = np.array([1, 1.5, 2, 2.5])
y = np.array([1.87640523, 2.59001572, 3.4978738 , 4.47408932])
yerr = 0.1

In [ ]:
plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k'); plt.show()

In [ ]:
def simulation(theta): 
    a, b = theta
    noise = yerr * np.random.normal(size=len(x))
    return a + b * x + noise

def sample_prior(): 
    # sample a and b from uniform prior
    _a = -0.5 + np.random.normal() 
    _b = 5 + 3*np.random.normal()
    return [_a, _b]

draw $(\theta', \mathcal{D}') \sim p(\theta, \mathcal{D})$ samples to train neural posterior estimator 

In [ ]:
Nsample = 10000

samples = np.zeros((Nsample, 6))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.concatenate([theta_p, data_p])
samples = torch.tensor(samples, dtype=torch.float32)

train a neural posterior estimator using MAF

In [ ]:
maf = NPE(density_estimator='maf')

# append (theta, D) from the simulations
_ = maf.append_simulations(samples[:,:2], samples[:,2:])
# train NPE. 
npe_maf = maf.train()

In [ ]:
npe_maf = maf.build_posterior()
q_samples = npe_maf.sample((10000,), x=torch.tensor(y, dtype=torch.float32))

In [ ]:
fig = plt.figure(figsize=(4,4))
fig = corner.corner(q_samples.numpy(), labels=['a', 'b'], fig=fig)
corner.overplot_lines(fig, [a_true, b_true], color="C1")
corner.overplot_points(fig, np.array([a_true, b_true])[None], marker="s", color="C1")

This looks like a reasonable posterior but we need to validate it more rigorously. 

# posterior predictive test
One common validation step is the posterior predictive check

$$\theta' \sim q_\phi(\theta|\mathcal{D}_{\rm obs})$$
then compare $F(\theta')$ to $\mathcal{D}_{\rm obs}$

In [ ]:
y_p = []
for theta_p in q_samples.numpy(): 
    y_p.append(simulation(theta_p))
y_p = np.array(y_p)

In [ ]:
plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k')
for _y in y_p[::100]: 
    plt.plot(x, _y, c='C0', lw=1, alpha=0.1)

Looks consistent but we want a more statistically rigorous validation using coverage tests

# construct test simulations
Draw $(\theta', \mathcal{D}') \sim p(\theta, \mathcal{D})$ using the same prior and simulator

In [ ]:
Ntest = 1000

theta_test = np.zeros((Ntest, 2))
data_test = np.zeros((Ntest, 4))
for i in range(Ntest): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    
    theta_test[i,:] = theta_p
    data_test[i,:] = data_p 

# simulation-based calibration (SBC)
Let's start with SBC

The algorithm is simple: 
```
for theta_test, data_test in test_samples: 

    # draw samples from posterior estimate
    q_samples = NPE(data_test)

    # compute rank 
    rank = sum(q_samples < theta_test)
```

In [ ]:
ranks_a = []
ranks_b = [] 

for _theta, _data in zip(theta_test, data_test): 
    _q_samples = npe_maf.sample((2000,), 
                                x=torch.tensor(_data, dtype=torch.float32), 
                                show_progress_bars=False)

    # calculate rank for each dimension
    ranks_a.append(np.sum(_q_samples.numpy()[:,0] < _theta[0])) 
    ranks_b.append(np.sum(_q_samples.numpy()[:,1] < _theta[1])) 

In [ ]:
fig = plt.figure(figsize=(8,2))
sub = fig.add_subplot(121)
_ = sub.hist(ranks_a, range=(0, 2000), bins=10)
sub.axhline(Ntest/10, color='k', linestyle='--')
sub.set_xlabel('ranks for $a$', fontsize=20)
sub.set_yticks([]); sub.set_xlim(0, 1000); sub.set_ylim(0., 1.3*Ntest/10)

sub = fig.add_subplot(122)
sub.hist(ranks_b, range=(0, 2000), bins=10)
sub.axhline(Ntest/10, color='k', linestyle='--')
sub.set_xlabel('ranks for $b$', fontsize=20)
sub.set_yticks([]); sub.set_xlim(0, 2000); sub.set_ylim(0., 1.3*Ntest/10)
plt.show()

SBC shows that $q_\phi$ is relatively well calibrated.

## probability-probability plot
For the PP plot, we only need to modify the rank statistic to estimate coverage

```
for theta_test, data_test in test_samples: 
    # drawn samples from posterior estimate
    q_samples = NPE(data_test)

    # estimate coverage 
    coverage = quantile(q_samples, theta_test)
```

In [ ]:
hist_a, _ = np.histogram(np.array(ranks_a) / 2000, 
                         range=(0., 1.), bins=100)
cdf_a = np.cumsum(hist_a) / Ntest

hist_b, _ = np.histogram(np.array(ranks_b) / 2000, 
                         range=(0., 1.), bins=100)
cdf_b = np.cumsum(hist_b) / Ntest

In [ ]:
fig = plt.figure(figsize=(8,3))
sub = fig.add_subplot(121)
sub.plot([0., 1.], [0., 1.], c='k', ls='--')
sub.plot(np.linspace(0., 1., 100), cdf_a)
sub.set_xlabel('expected coverage for $a$', fontsize=15)
sub.set_ylabel('estimated coverage', fontsize=15)
sub.set_xlim(0, 1.); sub.set_ylim(0., 1.)

sub = fig.add_subplot(122)
sub.plot([0., 1.], [0., 1.], c='k', ls='--')
sub.plot(np.linspace(0., 1., 100), cdf_b)
sub.set_xlabel('expected coverage for $b$', fontsize=15)
sub.set_xlim(0, 1.); sub.set_ylim(0., 1.); plt.show()

# Tests of Accuracy with Random Points (TARP) coverage
The validation above were for the 1D marginal posteriors for $a$ and $b$ separately. Now lets try TARP using their python package: [`tarp`](https://github.com/Ciela-Institute/tarp)

In [ ]:
import tarp

In [ ]:
q_samp = [] 
for _theta, _data in zip(theta_test, data_test): 
    _q_samples = npe_maf.sample((1000,), x=torch.tensor(_data, dtype=torch.float32), show_progress_bars=False)
    q_samp.append(_q_samples)
q_samp = np.array(q_samp)

In [ ]:
ecp, alpha = tarp.get_tarp_coverage(q_samp, theta_test, 
                                    references="random", 
                                    metric="euclidean")

In [ ]:
fig = plt.figure(figsize=(3,3))
sub = fig.add_subplot(111)
sub.plot([0., 1.], [0., 1.], c='k', ls='--')
sub.plot(alpha, ecp)
sub.set_xlabel('expected coverage', fontsize=15)
sub.set_ylabel('estimated coverage', fontsize=15)
sub.set_xlim(0, 1.); sub.set_ylim(0., 1.); plt.show()

This confirms that the MAF provides a well-calibrated posterior!

---
# In-class exercise: can we trust the IMF posterior?

Let's return to the IMF from last lecture: the slope $\alpha$ and the upper mass cutoff $m_{\rm max}$ of the IMF, from a cluster of 500 stars with masses between $1\,M_\odot$ and $m_{\rm max}$.

Same as before:

- $\theta = (\alpha, \log_{10} m_{\rm max})$, with priors $\alpha \sim \mathcal{N}(2.35, 0.3^2)$ and
  $\log_{10} m_{\rm max} \sim \mathcal{N}(1.9, 0.2^2)$;
- $\mathcal{D} = (\overline{\log m},\ \log m_{\rm most\ massive})$

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from torch.distributions import MultivariateNormal

torch.manual_seed(24)
rng = np.random.default_rng(24)

N_stars, m_lo = 500, 1.0
prior_mean, prior_std = np.array([2.35, 1.9]), np.array([0.3, 0.2])
prior = MultivariateNormal(torch.tensor(prior_mean, dtype=torch.float32),
                           torch.diag(torch.tensor(prior_std, dtype=torch.float32)**2))   # theta = (alpha, log10 m_max)

def sample_masses(alpha, m_max, n=N_stars):
    ''' inverse-CDF sampling of p(m) ~ m^-alpha on [m_lo, m_max] '''
    u = rng.random(n)
    a1 = 1.0 - alpha
    return (m_lo**a1 + u * (m_max**a1 - m_lo**a1))**(1.0 / a1)

def summarize(m):
    return np.array([np.mean(np.log(m)), np.log(m.max())])

def simulator(theta):
    ''' theta: [n, 2] array of (alpha, log10 m_max) -> D: [n, 2] tensor '''
    theta = np.atleast_2d(np.asarray(theta))
    return torch.tensor(np.array([summarize(sample_masses(a, 10**lM)) for a, lM in theta]), dtype=torch.float32)

def train_npe(n_sims):
    theta = prior.sample((n_sims,))
    npe = NPE(prior=prior, density_estimator='maf', show_progress_bars=False)
    npe.append_simulations(theta, simulator(theta.numpy()))
    npe.train(training_batch_size=min(256, n_sims // 4))
    return npe.build_posterior()

## 1. Train an NPE and validate it
Train NPE on 10,000 simulations. Then draw a **test set** of 500 new $(\theta', \mathcal{D}')$ pairs from the prior and simulator. 
Treat each test cluster, $(\theta_{i,{\rm test}}, \mathcal{D}_{i, {\rm test}})$, as an "observation" and 1000 samples from the posterior: $p(\theta\mid\mathcal{D}_{i,{\rm test}})$. 

## 2. Validate using Simulation-Based Calibration
Run the SBC test and plot the histograms for the two parameters

## PP plot
Generate PP plot for the two parameters

## TARP
Plot the TARP coverage of the joint distribution. 

How do the posterior validation look? 

## 3. is calibration enough? 
Validate three more posterior estimates with the same test set:
1. NPE trained on only 100 simulations
2. Prior --- ignore the data completely
3. an overconfident posterior. This one will be artificial. Take the 10,000-simulation NPE, sample it, then shrink the samples toward the mean by a factor of 0.5.

Compare the three posteriors using PP-plot and TARP. What do you find? 

## 4. model misspecification
Every test so far drew its test clusters from **our own simulator**. 
While coverage tests check that $q_\phi$ is a good approximation of the posterior, they cannot tell us whether the simulator describes the real universe.

We've assumed so far that we can detect every star but a real survey can be incomplete in the low-mass end. Suppose there's a mass dependent incompleteness: 50% at $1.3\,M_\odot$ and 90% at $1.7\,M_\odot$. 

Let's say the true IMF of the cluster is exactly our model, with $\alpha = 2.35$ and $m_{\rm max} = 80\,M_\odot$, but we only observe the stars the survey detected (`detected_cluster`). 
The simulator assumes every star is detected.

In [ ]:
def detected_cluster(alpha, m_max, m50=1.3, k=8, n=N_stars):
    ''' 500 *detected* stars; detection probability 1 / (1 + (m / m50)^-k) '''
    stars = []
    while len(stars) < n:
        m = sample_masses(alpha, m_max, n=4 * n)
        p_det = 1.0 / (1.0 + (m / m50)**-k)
        stars += list(m[rng.random(len(m)) < p_det])
    return np.array(stars[:n])

m_obs = detected_cluster(2.35, 80.0)
x_obs = torch.tensor(summarize(m_obs), dtype=torch.float32)

How does the 10k NPE posterior conditioned on `x_obs` compare to the truth? Plot the comparison. 

## posterior predictive check
Let's run a PPC on the IMF (full mass distribution), not just summaries the summaries. Use `sample_masses` on samples of the posterior to get the masses. Then compare to the sample masses from `detected_cluster`. 